In this code I attempt to perform the Olley-Pakes procedure to estimation production function 

In [1]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using Random
using Distributions
using Optim
using Plots
using ShiftedArrays  # for lag function

In [2]:
# Step 0: import and browse dataset
dataset = CSV.read("op_lp_ready.csv", DataFrame)

# first(dataset, 5)
# remove observations with zero invesment (satisfying invertibility condition)
data_filtered = filter(row -> row.v_investment > 0, dataset)

# data_1990 = filter(row -> row.year == 1990, data_investment_positive)

# # this data of the year 1990 will be one we use for our first estimation
# first(data_1990, 5)


Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64
1,1,1997,15.2437,12.2967,13.6469,14.7629,27.4075
2,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425
3,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536
4,2,1997,14.8474,12.3572,13.6883,14.2698,43.5369
5,2,1998,14.7601,11.5891,13.6968,14.2355,41.273
6,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179
7,3,1998,13.9124,12.1451,12.0143,13.4115,17.4865
8,3,1999,13.66,9.89273,12.0052,13.0285,17.4949
9,4,1995,17.6699,14.5626,17.1751,17.2448,71.6182


Now we proceed with step 1 of Olley-Pakes

I will fit a second-order polynomial on f(capital, material) and perform a regression with production (as dependent variable), labor, capital and investment to find beta_labor and the non-parametric fit.  


In [3]:
# A0: generating the polynomial terms 
data_filtered.v_capital_square = data_filtered.v_capital .* data_filtered.v_capital
data_filtered.v_investment_square = data_filtered.v_investment .* data_filtered.v_investment
data_filtered.int_capital_investment = data_filtered.v_capital .* data_filtered.v_investment

# display(first(data_1990, 5))

# A1: running the step 1 regression 
step1 = lm(@formula(v_production ~ v_labor + v_capital + v_capital_square + v_investment + v_investment_square + int_capital_investment), data_filtered)

display(step1) #print out the labor coefficient

β_labor = coef(step1)[2]

# B0: calculating the necessary variables for step 2 
# calculating residualized production without labor 
data_filtered.production_residuals = data_filtered.v_production - β_labor * data_filtered.v_labor 

# calculating predicted phi 
data_filtered.predicted_phi = (coef(step1)[1] .+ data_filtered.v_capital .* coef(step1)[3] .+
                             data_filtered.v_capital_square .* coef(step1)[4] .+ data_filtered.v_investment .* coef(step1)[5] .+ 
                             data_filtered.v_investment_square .* coef(step1)[6] .+
                             data_filtered.int_capital_investment .* coef(step1)[7])

# now we delete the old variables 
select!(data_filtered, Not([:v_investment_square, :v_capital_square]))

# B0: now we calculate the lagged phi needed for step 2: lag_phi and lag_capital 
sort!(data_filtered, [:firm_id, :year])
transform!(groupby(data_filtered, :firm_id), :v_capital => (x -> lag(x, 1)) => :lag_capital) #generate capital lag variable
transform!(groupby(data_filtered, :firm_id), :predicted_phi => (x -> lag(x, 1)) => :lag_phi) # generate phi lag variable
transform!(groupby(data_filtered, :firm_id), :v_investment => (x -> lag(x, 1)) => :lag_invesment) # generate phi lag variable

# then remove observations where we miss any one of the variables 
data_filtered = dropmissing(data_filtered, [:lag_capital, :lag_phi, :lag_invesment])

# display the dataset now ready for step 2's GMM
display(first(data_filtered, 10))

StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

v_production ~ 1 + v_labor + v_capital + v_capital_square + v_investment + v_investment_square + int_capital_investment

Coefficients:
────────────────────────────────────────────────────────────────────────────────────────────
                              Coef.   Std. Error      t  Pr(>|t|)     Lower 95%    Upper 95%
────────────────────────────────────────────────────────────────────────────────────────────
(Intercept)              6.39678     0.226873     28.20    <1e-99   5.95206       6.84149
v_labor                  0.0231743   0.000546768  42.38    <1e-99   0.0221025     0.024246
v_capital                0.498457    0.0516108     9.66    <1e-21   0.397289      0.599625
v_capital_square         0.00639769  0.00373071    1.71    0.0864  -0.000915275   0.0137107
v_investment             0.0569083   0.0361

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,int_capital_investment,production_residuals,predicted_phi,lag_capital,lag_phi,lag_invesment
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,1,1998,15.4399,12.6583,13.8412,15.0051,20.8425,175.206,14.9569,14.611,13.6469,14.4582,12.2967
2,1,1999,15.3679,12.3587,13.9283,14.9568,25.9536,172.136,14.7665,14.6062,13.8412,14.611,12.6583
3,2,1998,14.7601,11.5891,13.6968,14.2355,41.273,158.734,13.8036,14.3784,13.6883,14.488,12.3572
4,2,1999,14.7038,12.6857,13.8871,14.0906,42.8179,176.168,13.7115,14.6379,13.6968,14.3784,11.5891
5,3,1999,13.66,9.89273,12.0052,13.0285,17.4949,118.764,13.2546,13.3058,12.0143,13.6475,12.1451
6,4,1996,17.6065,15.2652,17.2278,17.2083,78.801,262.987,15.7803,16.7123,17.1751,16.5678,14.5626
7,5,1999,16.0031,14.0184,15.313,15.6114,51.9997,214.664,14.7981,15.5569,15.1472,15.3757,13.4246
8,6,1999,11.0108,8.07091,8.89698,9.20659,-4.84692,71.8067,11.1232,11.5321,8.54161,11.3673,8.14352
9,7,1997,12.5013,8.65869,9.51618,11.9876,16.5183,82.3977,12.1185,11.9108,9.18124,11.8331,9.26936


# Step 2: running the GMM estimation to find $\beta_k$ and Markov parameters

We decide on the moment conditions being of [$\epsilon$ | capital, lag_capital, lag_investment]



In [4]:
# B0: defining the crucial functions for our GMM procedure 

function markov(parameter_guess,
    lag_phi,
    lag_capital
)   
    beta_k = parameter_guess[1]
    alpha_0 = parameter_guess[2]
    alpha_1 = parameter_guess[3]

    # first-order markov process: now = alpha_0 + alpha_1 * last 
    value = alpha_0 .+ alpha_1 .* (lag_phi -  beta_k .*lag_capital)

    return value 
end 

# now onto the objective function 
function objective_function(
    parameter_guess,
    predicted_phi,
    lag_phi,
    capital, 
    lag_capital, 
    lag_investment 
)   
    beta_k = parameter_guess[1]
    alpha_0 = parameter_guess[2]
    alpha_1 = parameter_guess[3]

    # first, calculate step 2 equation's RHS:
    # RHS = beta_k * capital 
    RHS = beta_k * capital + markov(parameter_guess, lag_phi, lag_capital)

    # then the epsilon term would be the difference
    epsilon = predicted_phi - RHS

    # now construct the moment conditions that epsilon is exogenous to capital, lag_capital, and lag_investment 
    instrument = Matrix(hcat(ones(length(capital)),capital, lag_capital, lag_investment))  # Nx4 matrix
    g = (transpose(epsilon) * instrument) ./ length(instrument)
    loss = dot(g, g)  # equivalent to g'T * g
    return loss # returning loss
end 

objective_function (generic function with 1 method)

In [5]:
# tester function 
objective_function([0,0,0],
    data_filtered.predicted_phi,
    data_filtered.lag_phi,
    data_filtered.v_capital,
    data_filtered.lag_capital,
    data_filtered.lag_invesment
)

4905.31932984009

In [6]:
# function to run the GMM procedure 
function GMM_main(
    initial_guess, 
    dataset
)
    result = optimize(parameter_guess -> objective_function(
            parameter_guess,
            dataset.predicted_phi,
            dataset.lag_phi,
            dataset.v_capital,
            dataset.lag_capital,
            dataset.lag_invesment
        ),
        initial_guess,
        NelderMead()
    )
    min_loss = Optim.minimum(result)   # minimum loss value
    println("Minimum loss value: ", min_loss)


    parameter_estimated = Optim.minimizer(result)
    println(parameter_estimated)
    println("Estimated capital coefficient: ", parameter_estimated[1])
end 

GMM_main([0.1,0.1,0.1], data_filtered)

println("Estimated labor coefficient: ", β_labor)
println("Mean productivity values: ", mean(data_filtered.predicted_phi))


Minimum loss value: 0.000950991570359477
[1.439934967153734, -0.4081606311592877, 0.9293746608963002]
Estimated capital coefficient: 1.439934967153734
Estimated labor coefficient: 0.02317426009464081
Mean productivity values: 13.588094251644584
